# Re-ID measurement: full pipeline on the two long HILAL-HAZM windows

Runs the current pipeline (our people + ball models, Roboflow v10 keypoints run locally, fixed team
classification) on `MyDrive/Playbook/benchmark/HIL-HAZ_half1.mp4` and `HIL-HAZ_half2.mp4`, the windows
whose player IDs were unified by hand earlier. Claude then scores identity over minutes (not 20 s) and
tunes the automatic re-linking and the merge list on it. No model training.

**Runtime:** T4 or L4 GPU, about 30-60 minutes. **Runtime -> Run all**; do not edit cells. If it
disconnects, Run all again: finished windows are skipped. Results: `MyDrive/Playbook/reid_windows/`.

In [ ]:
# Cell 1 — GPU, Drive, settings
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = f'{PBD}/reid_windows'
os.makedirs(DEST, exist_ok=True)
WINDOWS = {'half1': f'{PBD}/benchmark/HIL-HAZ_half1.mp4', 'half2': f'{PBD}/benchmark/HIL-HAZ_half2.mp4'}
NEEDED = [f'{PBD}/people_model/train/checkpoint_best_total.pth', f'{PBD}/ball_model/train/checkpoint_best_total.pth',
          f'{PBD}/field_model/roboflow_v10/weights.pt'] + list(WINDOWS.values())
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit(f'Missing in Drive: {missing}')
print('all inputs found')

In [ ]:
# Cell 2 — Our repo and packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
os.chdir(PB)
!pip install -q "rfdetr[train]==1.11.0" "git+https://github.com/roboflow/sports.git@main" "pydantic-settings==2.4.0" "python-dotenv==1.0.1" "lap>=0.5.13,<0.6"
shutil.copy('baseline.env', '.env')  # the pilot setup (model paths point to MyDrive/Playbook)
print('ready')

In [ ]:
# Cell 3 — Run the pipeline on both windows (skips finished ones)
for name, video in WINDOWS.items():
    out = f'{DEST}/{name}'
    if os.path.exists(f'{out}/kpi_summary.json'):
        print(name, 'already done'); continue
    local = f'/content/run_{name}'
    shutil.rmtree(local, ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, 'main.py', '--source-video', video, '--out-dir', local, '--no-video',
                        '--enable-team', '--record-cache', f'{local}/cache'], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
    if r.returncode != 0:
        raise SystemExit(f'{name} failed: send Claude a screenshot of the lines above.')
    shutil.copytree(local, out, dirs_exist_ok=True)
    print(f'{name}: done in {(time.time() - t0) / 60:.0f} min')
print(f'\nDone. Tell Claude the results are in {DEST}.')